# GroupBy: lo que probablemente no conoces

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/14_pandas_transformaciones/code/03_groupby.ipynb)

---

Este notebook **no es una intro a groupby** — ya viste eso en cursos anteriores.

El objetivo es cubrir las partes que la mayoría no conoce y que son las más útiles en la práctica:

- **Named aggregations** — resultado limpio sin MultiIndex raro
- **`transform()`** — la operación que más se subestima
- **`filter()`** — filtrar grupos enteros, no filas individuales
- **`apply()`** con groupby — el martillo cuando todo lo demás falla
- Patrones comunes en ciencia de datos

Si ya dominas todo esto, usa el notebook como referencia rápida.

In [1]:
%pip install pandas==2.2.3 numpy==1.26.4

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np

np.random.seed(42)
df = pd.DataFrame({
    "empleado_id": range(1, 21),
    "nombre": [f"Emp_{i}" for i in range(1, 21)],
    "departamento": np.random.choice(["Ventas", "Tech", "RRHH", "Finanzas"], 20),
    "ciudad": np.random.choice(["CDMX", "MTY", "GDL"], 20),
    "salario": np.random.randint(25000, 90000, 20),
    "años_experiencia": np.random.randint(1, 15, 20),
    "activo": np.random.choice([True, False], 20, p=[0.8, 0.2]),
})

print(f"Shape: {df.shape}")
df.head(10)

Shape: (20, 7)


,empleado_id,nombre,departamento,ciudad,salario,años_experiencia,activo
0,1,Emp_1,RRHH,MTY,44118,2,True
1,2,Emp_2,Finanzas,CDMX,60773,10,True
2,3,Emp_3,Ventas,MTY,78810,9,True
3,4,Emp_4,RRHH,MTY,26899,10,True
4,5,Emp_5,RRHH,MTY,81886,5,False
5,6,Emp_6,Finanzas,MTY,26267,2,True
6,7,Emp_7,Ventas,CDMX,56551,4,False
7,8,Emp_8,Ventas,CDMX,75680,12,False
8,9,Emp_9,RRHH,MTY,36394,12,True
9,10,Emp_10,Tech,MTY,28556,7,False


## Sección 1: Breve recap — `agg()` ya lo conoces

Repaso rápido para alinear el vocabulario antes de ver lo nuevo.

In [3]:
# Forma básica: una sola estadística
df.groupby("departamento")["salario"].mean()

departamento
Finanzas    44047.333333
RRHH        54865.111111
Tech        28556.000000
Ventas      64454.500000
Name: salario, dtype: float64

In [4]:
# Varias estadísticas a la vez
df.groupby("departamento")["salario"].agg(["mean", "std", "count"])

,mean,std,count
departamento,,,
Finanzas,44047.333333,16624.861511,6
RRHH,54865.111111,22015.505340,9
Tech,28556.000000,NaN,1
Ventas,64454.500000,15351.998230,4


In [5]:
# Agrupación por múltiples columnas
df.groupby(["departamento", "ciudad"])["salario"].mean()

departamento  ciudad
Finanzas      CDMX      60773.000000
              GDL       36647.000000
              MTY       43405.666667
RRHH          CDMX      50412.000000
              GDL       85232.000000
              MTY       51463.600000
Tech          MTY       28556.000000
Ventas        CDMX      66115.500000
              GDL       46777.000000
              MTY       78810.000000
Name: salario, dtype: float64

**Punto clave:** `.agg()` *colapsa* los datos — el resultado tiene **una fila por grupo**, no una por empleado.

> 💡 **Prueba esto:** ¿Cuántas filas tiene cada resultado arriba vs el DataFrame original (20 filas)?

In [6]:
print("Filas resultado agrupado:", df.groupby(["departamento", "ciudad"]).agg({"salario": "mean"}).shape[0])
print("Filas DataFrame original:", df.shape[0])

Filas resultado agrupado: 10
Filas DataFrame original: 20


## Sección 2: Named aggregations — el resultado limpio

Cuando agrupas por múltiples columnas con el diccionario clásico, pandas genera columnas jerárquicas (MultiIndex) que son difíciles de manejar.

In [7]:
# Forma antigua — columnas jerárquicas raras
resultado_feo = df.groupby("departamento").agg(
    {"salario": ["mean", "max"], "años_experiencia": "mean"}
)
print("Columnas:", resultado_feo.columns.tolist())
resultado_feo

Columnas: [('salario', 'mean'), ('salario', 'max'), ('años_experiencia', 'mean')]


salario        años_experiencia
                      mean    max             mean
departamento                                      
Finanzas      44047.333333  68323         6.666667
RRHH          54865.111111  85232         7.666667
Tech          28556.000000  28556         7.000000
Ventas        64454.500000  78810         6.500000

In [8]:
# Named aggregations (pandas >= 0.25) — columnas con nombres que tú eliges
resultado = df.groupby("departamento").agg(
    salario_promedio=pd.NamedAgg(column="salario", aggfunc="mean"),
    salario_max=pd.NamedAgg(column="salario", aggfunc="max"),
    experiencia_promedio=pd.NamedAgg(column="años_experiencia", aggfunc="mean"),
    n_empleados=pd.NamedAgg(column="empleado_id", aggfunc="count"),
)

print("Columnas:", resultado.columns.tolist())
resultado

Columnas: ['salario_promedio', 'salario_max', 'experiencia_promedio', 'n_empleados']


,salario_promedio,salario_max,experiencia_promedio,n_empleados
departamento,,,,
Finanzas,44047.333333,68323,6.666667,6
RRHH,54865.111111,85232,7.666667,9
Tech,28556.000000,28556,7.000000,1
Ventas,64454.500000,78810,6.500000,4


Sin MultiIndex, sin columnas raras. El resultado es un DataFrame normal que puedes usar directamente.

También puedes usar lambdas como `aggfunc`:

> 💡 **Prueba esto:** Agrega una columna `porcentaje_activos` al resultado usando `aggfunc=lambda x: x.mean()` sobre la columna `activo` (que es bool, así que `.mean()` da la proporción de `True`).

In [9]:
df.groupby("departamento").agg(
    salario_promedio=("salario", "mean"),
    salario_max=("salario", "max"),
    salario_min=("salario", "min"),
    porcentaje_activos=("activo", lambda x: x.mean())
)

,salario_promedio,salario_max,salario_min,porcentaje_activos
departamento,,,,
Finanzas,44047.333333,68323,26267,0.833333
RRHH,54865.111111,85232,26899,0.888889
Tech,28556.000000,28556,28556,0.000000
Ventas,64454.500000,78810,46777,0.500000


## Sección 3: `transform()` — el que no conocen

Esta es la operación que más se subestima y más se usa en feature engineering.

**La diferencia fundamental con `agg()`:**
- `agg()` → colapsa a un valor por grupo (el resultado tiene menos filas)
- `transform()` → devuelve una Serie con la **misma longitud** que el DataFrame original

In [10]:
# agg: colapsa a un valor por grupo
medias_agg = df.groupby("departamento")["salario"].agg("mean")
print(f"agg shape: {medias_agg.shape}")  # (4,) — solo 4 departamentos
print(medias_agg)
print()

agg shape: (4,)
departamento
Finanzas    44047.333333
RRHH        54865.111111
Tech        28556.000000
Ventas      64454.500000
Name: salario, dtype: float64



In [11]:
# transform: mantiene todos los registros originales
medias_transform = df.groupby("departamento")["salario"].transform("mean")
print(f"transform shape: {medias_transform.shape}")  # (20,) — todos los empleados
print(medias_transform.head(10))

transform shape: (20,)
0    54865.111111
1    44047.333333
2    64454.500000
3    54865.111111
4    54865.111111
5    44047.333333
6    64454.500000
7    64454.500000
8    54865.111111
9    28556.000000
Name: salario, dtype: float64


In [12]:
# Esto lo hace perfecto para agregar columnas nuevas al DataFrame original
df["salario_medio_depto"] = df.groupby("departamento")["salario"].transform("mean")

# Cada empleado ahora tiene el salario promedio de SU departamento
df[["nombre", "departamento", "salario", "salario_medio_depto"]].head(10)

,nombre,departamento,salario,salario_medio_depto
0,Emp_1,RRHH,44118,54865.111111
1,Emp_2,Finanzas,60773,44047.333333
2,Emp_3,Ventas,78810,64454.500000
3,Emp_4,RRHH,26899,54865.111111
4,Emp_5,RRHH,81886,54865.111111
5,Emp_6,Finanzas,26267,44047.333333
6,Emp_7,Ventas,56551,64454.500000
7,Emp_8,Ventas,75680,64454.500000
8,Emp_9,RRHH,36394,54865.111111
9,Emp_10,Tech,28556,28556.000000


**Por qué importa en ML:** Necesitas features a nivel fila, no a nivel grupo. `transform()` te permite crear features como "¿cuánto gana este empleado comparado con su departamento?" sin perder las filas originales.

In [13]:
# Normalización dentro de grupo (z-score por departamento)
df["salario_norm_depto"] = (
    df["salario"] - df.groupby("departamento")["salario"].transform("mean")
) / df.groupby("departamento")["salario"].transform("std")

# Rank dentro de departamento (1 = el que menos gana en su depto)
df["salario_rank_depto"] = df.groupby("departamento")["salario"].transform("rank")

# Booleano: ¿gana más que la media de su departamento?
df["es_mayor_media"] = df["salario"] > df.groupby("departamento")["salario"].transform("mean")

df[["nombre", "departamento", "salario", "salario_norm_depto", "salario_rank_depto", "es_mayor_media"]].head(10)

,nombre,departamento,salario,salario_norm_depto,salario_rank_depto,es_mayor_media
0,Emp_1,RRHH,44118,-0.488161,4.0,False
1,Emp_2,Finanzas,60773,1.006064,5.0,True
2,Emp_3,Ventas,78810,0.935090,4.0,True
3,Emp_4,RRHH,26899,-1.270292,1.0,False
4,Emp_5,RRHH,81886,1.227357,8.0,True
5,Emp_6,Finanzas,26267,-1.069503,1.0,False
6,Emp_7,Ventas,56551,-0.514819,2.0,False
7,Emp_8,Ventas,75680,0.731208,3.0,True
8,Emp_9,RRHH,36394,-0.839005,3.0,False
9,Emp_10,Tech,28556,NaN,1.0,False


> 💡 **Prueba esto:** ¿Puedes crear una columna `años_experiencia_pct` que sea la experiencia del empleado dividida entre la experiencia promedio de su departamento? (valores > 1 = más experiencia que el promedio del depto)

In [15]:
df["años_experiencia_pct"] = df["años_experiencia"] / df.groupby("departamento")["años_experiencia"].transform("mean")

print(df[["nombre", "departamento", "años_experiencia", "años_experiencia_pct"]])

    nombre departamento  años_experiencia  años_experiencia_pct
0    Emp_1         RRHH                 2              0.260870
1    Emp_2     Finanzas                10              1.500000
2    Emp_3       Ventas                 9              1.384615
3    Emp_4         RRHH                10              1.304348
4    Emp_5         RRHH                 5              0.652174
5    Emp_6     Finanzas                 2              0.300000
6    Emp_7       Ventas                 4              0.615385
7    Emp_8       Ventas                12              1.846154
8    Emp_9         RRHH                12              1.565217
9   Emp_10         Tech                 7              1.000000
10  Emp_11         RRHH                12              1.565217
11  Emp_12         RRHH                13              1.695652
12  Emp_13         RRHH                 8              1.043478
13  Emp_14         RRHH                 3              0.391304
14  Emp_15     Finanzas                1

## Sección 4: `filter()` — filtrar grupos enteros

`filter()` no filtra filas individuales — filtra **grupos completos**.

La función recibe un grupo (DataFrame) y debe devolver `True` (conservar todo el grupo) o `False` (descartar todo el grupo).

In [16]:
# Mantener solo departamentos con salario promedio > 50,000
df_filtrado = df.groupby("departamento").filter(
    lambda grupo: grupo["salario"].mean() > 50000
)

print(f"Filas originales:          {len(df)}")
print(f"Filas después de filter:   {len(df_filtrado)}")
print(f"Departamentos restantes:   {df_filtrado['departamento'].unique()}")

Filas originales:          20
Filas después de filter:   13
Departamentos restantes:   ['RRHH' 'Ventas']


In [17]:
# Diferencia importante: filter() vs query()

# query filtra FILA POR FILA — un empleado puede quedar aunque su depto tenga media baja
df_query = df.query("salario > 50000")
print(f"query (fila por fila):     {len(df_query)} filas")

# filter filtra GRUPOS ENTEROS — si el depto no cumple, pierdes a TODOS sus empleados
df_filter = df.groupby("departamento").filter(lambda g: g["salario"].mean() > 50000)
print(f"filter (grupo completo):   {len(df_filter)} filas")
print()
print("query conserva empleados con salario > 50k aunque su depto no pase el corte.")
print("filter elimina a TODOS los empleados de deptos que no cumplen la condición.")

query (fila por fila):     10 filas
filter (grupo completo):   13 filas

query conserva empleados con salario > 50k aunque su depto no pase el corte.
filter elimina a TODOS los empleados de deptos que no cumplen la condición.


> 💡 **Prueba esto:** Filtra para conservar solo los departamentos que tienen **más de 5 empleados activos** (`activo == True`).

In [19]:
activos_por_depto = df.groupby("departamento")["activo"].sum()

deptos_validos = activos_por_depto[activos_por_depto > 5].index

df_mas_5_activos = df[df["departamento"].isin(deptos_validos)]

print(df_mas_5_activos)
print("\nDepartamentos restantes:", df_mas_5_activos["departamento"].unique())

    empleado_id  nombre departamento ciudad  salario  años_experiencia  \
0             1   Emp_1         RRHH    MTY    44118                 2   
3             4   Emp_4         RRHH    MTY    26899                10   
4             5   Emp_5         RRHH    MTY    81886                 5   
8             9   Emp_9         RRHH    MTY    36394                12   
10           11  Emp_11         RRHH   CDMX    28890                12   
11           12  Emp_12         RRHH   CDMX    66606                13   
12           13  Emp_13         RRHH   CDMX    55740                 8   
13           14  Emp_14         RRHH    GDL    85232                 3   
19           20  Emp_20         RRHH    MTY    68021                 4   

    activo  salario_medio_depto  salario_norm_depto  salario_rank_depto  \
0     True         54865.111111           -0.488161                 4.0   
3     True         54865.111111           -1.270292                 1.0   
4    False         54865.111111   

## Sección 5: `apply()` con groupby — el más flexible (y el más lento)

Cuando `agg`, `transform` y `filter` no alcanzan, existe `apply`.

La función recibe un grupo (DataFrame) y puede devolver lo que quieras: un escalar, una Serie, otro DataFrame.

In [20]:
# Caso que no se puede hacer fácil con agg: necesito columnas múltiples en la salida
def top_empleados(grupo, n=2):
    """Devuelve los n empleados con mayor salario en el grupo."""
    return grupo.nlargest(n, "salario")[["nombre", "salario"]]

# include_groups=False evita un DeprecationWarning en pandas >= 2.2
df.groupby("departamento").apply(top_empleados, n=2, include_groups=False)

nombre  salario
departamento                    
Finanzas     18  Emp_19    68323
             1    Emp_2    60773
RRHH         13  Emp_14    85232
             4    Emp_5    81886
Tech         9   Emp_10    28556
Ventas       2    Emp_3    78810
             7    Emp_8    75680

**Advertencia:** `apply` es significativamente más lento que `agg`, `transform` y `filter` porque no puede vectorizarse igual. Úsalo solo cuando las otras opciones no funcionan.

> 💡 **Prueba esto:** ¿Puedes obtener el mismo resultado (top 2 por departamento) sin `groupby.apply`? Pista: `sort_values` + `groupby` + `.head(2)`.

In [28]:
df_top2 = (
    df.sort_values("salario", ascending=False)
      .groupby("departamento")[["nombre", "salario"]]
      .head(2)
)

print(df_top2)

    nombre  salario
13  Emp_14    85232
4    Emp_5    81886
2    Emp_3    78810
7    Emp_8    75680
18  Emp_19    68323
1    Emp_2    60773
9   Emp_10    28556


## Sección 6: Patrones comunes en ciencia de datos

Estos son los patrones que vas a usar constantemente. Memorízalos o guarda este notebook como referencia.

In [29]:
# Conteo por combinación de grupos
df.groupby(["departamento", "ciudad"]).size().reset_index(name="count")

,departamento,ciudad,count
0,Finanzas,CDMX,1
1,Finanzas,GDL,2
2,Finanzas,MTY,3
3,RRHH,CDMX,3
4,RRHH,GDL,1
5,RRHH,MTY,5
6,Tech,MTY,1
7,Ventas,CDMX,2
8,Ventas,GDL,1
9,Ventas,MTY,1


In [30]:
# Proporción del salario de cada empleado dentro de su departamento
total_por_depto = df.groupby("departamento")["salario"].transform("sum")
df["prop_salario"] = df["salario"] / total_por_depto

# Verificación: la proporción dentro de cada depto debe sumar 1
df.groupby("departamento")["prop_salario"].sum()

departamento
Finanzas    1.0
RRHH        1.0
Tech        1.0
Ventas      1.0
Name: prop_salario, dtype: float64

In [31]:
# Primer/último registro por grupo después de ordenar
# Útil para: primer evento por usuario, registro más reciente, etc.
df.sort_values("salario").groupby("departamento").first()[["nombre", "salario", "años_experiencia"]]

,nombre,salario,años_experiencia
departamento,,,
Finanzas,Emp_6,26267,2
RRHH,Emp_4,26899,10
Tech,Emp_10,28556,7
Ventas,Emp_16,46777,1


In [32]:
# Suma acumulada dentro de grupo
# Útil para: acumulado de ventas por vendedor, acumulado por mes, etc.
df_sorted = df.sort_values(["departamento", "salario"]).copy()
df_sorted["salario_acumulado_depto"] = (
    df_sorted.groupby("departamento")["salario"].transform("cumsum")
)
df_sorted[["departamento", "nombre", "salario", "salario_acumulado_depto"]].head(12)

,departamento,nombre,salario,salario_acumulado_depto
5,Finanzas,Emp_6,26267,26267
17,Finanzas,Emp_18,33792,60059
16,Finanzas,Emp_17,35627,95686
14,Finanzas,Emp_15,39502,135188
1,Finanzas,Emp_2,60773,195961
18,Finanzas,Emp_19,68323,264284
3,RRHH,Emp_4,26899,26899
10,RRHH,Emp_11,28890,55789
8,RRHH,Emp_9,36394,92183
0,RRHH,Emp_1,44118,136301


## Sección 7: Ejercicio

Usa el DataFrame `df` con el que trabajamos todo el notebook. Los ejercicios van en orden de dificultad.

**Ejercicio 1 — Named aggregations**

Crea un resumen por departamento con las siguientes columnas:
- `salario_promedio` — promedio de salario
- `salario_max` — salario máximo
- `n_empleados` — número de empleados
- `pct_activos` — porcentaje de empleados activos (entre 0 y 1)

El resultado debe ser un DataFrame limpio sin MultiIndex.

In [33]:
# Tu solución aquí
df_resumen = df.groupby("departamento").agg(
    salario_promedio=("salario", "mean"),
    salario_max=("salario", "max"),
    n_empleados=("salario", "size"),
    pct_activos=("activo", "mean")
).reset_index()

print(df_resumen)

  departamento  salario_promedio  salario_max  n_empleados  pct_activos
0     Finanzas      44047.333333        68323            6     0.833333
1         RRHH      54865.111111        85232            9     0.888889
2         Tech      28556.000000        28556            1     0.000000
3       Ventas      64454.500000        78810            4     0.500000


**Ejercicio 2 — transform()**

Agrega una columna `salario_percentil_depto` que indique el percentile rank del salario de cada empleado **dentro de su departamento** (entre 0 y 1).

Pista: `transform` acepta funciones personalizadas. Busca `pd.Series.rank` con `pct=True`.

In [34]:
# Tu solución aquí
df["salario_percentil_depto"] = df.groupby("departamento")["salario"].transform(
    "rank", pct=True
)

print(df[["nombre", "departamento", "salario", "salario_percentil_depto"]])

    nombre departamento  salario  salario_percentil_depto
0    Emp_1         RRHH    44118                 0.444444
1    Emp_2     Finanzas    60773                 0.833333
2    Emp_3       Ventas    78810                 1.000000
3    Emp_4         RRHH    26899                 0.111111
4    Emp_5         RRHH    81886                 0.888889
5    Emp_6     Finanzas    26267                 0.166667
6    Emp_7       Ventas    56551                 0.500000
7    Emp_8       Ventas    75680                 0.750000
8    Emp_9         RRHH    36394                 0.333333
9   Emp_10         Tech    28556                 1.000000
10  Emp_11         RRHH    28890                 0.222222
11  Emp_12         RRHH    66606                 0.666667
12  Emp_13         RRHH    55740                 0.555556
13  Emp_14         RRHH    85232                 1.000000
14  Emp_15     Finanzas    39502                 0.666667
15  Emp_16       Ventas    46777                 0.250000
16  Emp_17    

**Ejercicio 3 — filter()**

Filtra el DataFrame para conservar solo los departamentos donde **todos** los empleados están activos (`activo == True`).

¿Cuántos departamentos quedan? ¿Cuántas filas?

In [35]:
# Tu solución aquí
df_filtrado = df.groupby("departamento").filter(
    lambda g: g["activo"].all()
)

print(df_filtrado)
print("\nDepartamentos restantes:", df_filtrado["departamento"].unique())
print("Filas:", len(df_filtrado))

Empty DataFrame
Columns: [empleado_id, nombre, departamento, ciudad, salario, años_experiencia, activo, salario_medio_depto, salario_norm_depto, salario_rank_depto, es_mayor_media, años_experiencia_pct, prop_salario, salario_percentil_depto]
Index: []

Departamentos restantes: []
Filas: 0


**Ejercicio 4 — transform() avanzado**

Crea una columna `nivel` con dos categorías:
- `"senior"` si el salario del empleado es mayor que el percentil 75 de su departamento
- `"junior"` en cualquier otro caso

Pista: Primero calcula el p75 por departamento con `transform`, luego compara.

In [36]:
# Tu solución aquí
p75 = df.groupby("departamento")["salario"].transform(lambda x: x.quantile(0.75))

df["nivel"] = (df["salario"] > p75).map({True: "senior", False: "junior"})

print(df[["nombre", "departamento", "salario", "nivel"]])

    nombre departamento  salario   nivel
0    Emp_1         RRHH    44118  junior
1    Emp_2     Finanzas    60773  senior
2    Emp_3       Ventas    78810  senior
3    Emp_4         RRHH    26899  junior
4    Emp_5         RRHH    81886  senior
5    Emp_6     Finanzas    26267  junior
6    Emp_7       Ventas    56551  junior
7    Emp_8       Ventas    75680  junior
8    Emp_9         RRHH    36394  junior
9   Emp_10         Tech    28556  junior
10  Emp_11         RRHH    28890  junior
11  Emp_12         RRHH    66606  junior
12  Emp_13         RRHH    55740  junior
13  Emp_14         RRHH    85232  senior
14  Emp_15     Finanzas    39502  junior
15  Emp_16       Ventas    46777  junior
16  Emp_17     Finanzas    35627  junior
17  Emp_18     Finanzas    33792  junior
18  Emp_19     Finanzas    68323  senior
19  Emp_20         RRHH    68021  junior
